In [0]:
%run /Users/aguschaud@gmail.com/utn-spark/tp-integrador/00_ads_data_generator

%md
# Generador de datos ficticios – Dominio Ads / Publicidad Digital

Esta notebook genera datos sintéticos pero estructuralmente realistas del dominio de **publicidad digital (AdTech)**.  

---

# 1. ¿Cómo funciona la publicidad digital?

En términos simplificados, el ecosistema funciona así:

1. Una **empresa (Advertiser)** quiere promocionar un producto o servicio.
2. Define una **Campaign (campaña)** con:
   - un objetivo (awareness, tráfico, conversiones, etc.),
   - un presupuesto,
   - un rango de fechas,
   - uno o más canales (Search, Social, Display, Video, etc.).
3. Dentro de cada campaña se crean **Ads (anuncios)**:
   - creatividades específicas (imagen, video, texto),
   - con una segmentación determinada (audiencia target).
4. Cuando un usuario navega por una app o sitio, el sistema decide mostrar un anuncio.
   - Ese evento se registra como una **Impression**.
5. Si el usuario hace clic en el anuncio:
   - se registra un **Click**.
6. Si luego realiza una acción valiosa (compra, registro, descarga):
   - se registra una **Conversion**.

De forma simplificada:

**Advertiser → Campaign → Ad → Impression → Click → Conversion**

Este flujo representa un embudo clásico de marketing digital.

---

# 2. Modelo de datos

Esta notebook modela el dominio con **6 entidades principales**:

## 1. advertisers
Empresas que invierten en publicidad.

## 2. campaigns
Campañas asociadas a un advertiser.
Contienen presupuesto, fechas, objetivo y canal.

## 3. ads
Creatividades individuales que pertenecen a una campaña.

## 4. impressions
Evento: el anuncio fue mostrado.

## 5. clicks
Evento: el usuario hizo clic.
Es un subconjunto de las impresiones.

## 6. conversions
Evento: acción valiosa posterior al click.
Es un subconjunto de los clicks.

# 3. Uso de las funciones

Cada entidad tiene una función con prefijo `get_`.

Funciones disponibles:

- `get_advertisers()`
- `get_campaigns()`
- `get_ads()`
- `get_impressions()`
- `get_clicks()`
- `get_conversions()`

## Comportamiento general

- Todas devuelven `list[dict]`.
- No tienen parámetros obligatorios.
- Son deterministas (mismos parámetros → mismo resultado).
- El parámetro `seed` es opcional y tiene un valor por defecto fijo.

---

## Eventos (impressions, clicks, conversions)

Parámetros disponibles:

- `start_date` (formato `"YYYY-MM-DD"`)
- `end_date` (formato `"YYYY-MM-DD"`)

Si no se indican fechas:
- Se generan datos correspondientes a **ayer (UTC)**.

Si se indica solo `start_date`:
- Se generan datos únicamente para ese día.

Si se indican ambas fechas:
- Se generan datos para todo el rango (inclusive).

---

Modificar fechas produce un dataset distinto, siempre de forma determinista.

In [0]:
%python
import json
import os
from datetime import datetime, timedelta, timezone, date

# 1. PARÁMETROS DE ENTRADA (widgets de la notebook).
#    Parametrizar = cambiar la fecha o la ruta SIN editar código:
#    el mismo notebook sirve para la carga diaria o para un backfill histórico.
#    - base_path : dónde aterrizar los archivos dentro del volumen.
#    - start_date / end_date: rango de fechas a generar (ambos inclusive).
#      Ambos vacíos = ayer (UTC), el default diario; si se indica uno solo, ese único día.
dbutils.widgets.text("base_path", "/Volumes/dev/bronze_ads/landing_ads", "Ruta base del volumen")
dbutils.widgets.text("start_date", "", "Fecha inicial (YYYY-MM-DD, vacío = ayer UTC)")
dbutils.widgets.text("end_date", "", "Fecha final (YYYY-MM-DD, vacío = igual a start_date)")

base_path = dbutils.widgets.get("base_path")
start_param = dbutils.widgets.get("start_date").strip()
end_param = dbutils.widgets.get("end_date").strip()

# date.fromisoformat valida el formato YYYY-MM-DD: si está mal escrito, el error aparece acá, 
# con un mensaje claro, y no a mitad de la generación.
ayer = datetime.now(timezone.utc).date() - timedelta(days=1)

if not start_param and not end_param:
    # default: ambos vacíos => solo ayer (la carga diaria).
    fecha_inicio = fecha_fin = ayer
else:
    # si falta un extremo, se completa con el otro => rango de un solo día.
    fecha_inicio = date.fromisoformat(start_param) if start_param else date.fromisoformat(end_param)
    fecha_fin = date.fromisoformat(end_param) if end_param else fecha_inicio

if fecha_inicio > fecha_fin:
    # falla acá con un mensaje claro en vez de generar un rango invertido/vacío.
    raise ValueError(f"start_date ({fecha_inicio}) no puede ser posterior a end_date ({fecha_fin})")

# versión compacta para el nombre del archivo: un día => esa fecha; rango => inicio_fin
if fecha_inicio == fecha_fin:
    fecha_archivo = fecha_inicio.strftime("%Y%m%d")
    print(f"Generando datos del día: {fecha_inicio.isoformat()}")
else:
    fecha_archivo = f"{fecha_inicio.strftime('%Y%m%d')}_{fecha_fin.strftime('%Y%m%d')}"
    print(f"Generando datos del rango: {fecha_inicio.isoformat()} a {fecha_fin.isoformat()} ({(fecha_fin - fecha_inicio).days + 1} días)")

# 2. Generar los datos crudos con las funciones de 00_ads_data_generator.
#    Todas devuelven list[dict] (Python puro), NO DataFrames de Pandas.
#    Dimensiones (advertisers/campaigns/ads) no dependen de la fecha;
#    eventos (impressions/clicks/conversions) sí: se generan para el rango elegido (inclusive).
rango = {"start_date": fecha_inicio.isoformat(), "end_date": fecha_fin.isoformat()}

entidades = {
    "advertisers":  get_advertisers(),
    "campaigns":    get_campaigns(),
    "ads":          get_ads(),
    "impressions":  get_impressions(**rango),
    "clicks":       get_clicks(**rango),
    "conversions":  get_conversions(**rango),
}

# 3. Escribir cada entidad como JSON Lines (1 registro JSON por línea).
#    Formato uniforme para toda la capa bronze y fácil de leer con COPY INTO.
for nombre, registros in entidades.items():
    carpeta = f"{base_path}/{nombre}"
    os.makedirs(carpeta, exist_ok=True)          # crea el subdirectorio si no existe

    archivo = f"{carpeta}/{nombre}_{fecha_archivo}.jsonl"
    with open(archivo, "w", encoding="utf-8") as f:
        for registro in registros:
            f.write(json.dumps(registro) + "\n")  # convierte dicts a texto, cada registro en su propia línea para que Copy Into lo lea bien

    print(f"{nombre}: {len(registros)} registros -> {archivo}")

Generando datos del día: 2026-11-08
advertisers: 200 registros -> /Volumes/dev/bronze_ads/landing_ads/advertisers/advertisers_20261108.jsonl
campaigns: 2000 registros -> /Volumes/dev/bronze_ads/landing_ads/campaigns/campaigns_20261108.jsonl
ads: 500000 registros -> /Volumes/dev/bronze_ads/landing_ads/ads/ads_20261108.jsonl
impressions: 5023 registros -> /Volumes/dev/bronze_ads/landing_ads/impressions/impressions_20261108.jsonl
clicks: 100 registros -> /Volumes/dev/bronze_ads/landing_ads/clicks/clicks_20261108.jsonl
conversions: 8 registros -> /Volumes/dev/bronze_ads/landing_ads/conversions/conversions_20261108.jsonl


In [0]:
-- INGESTA A LA CAPA BRONZE CON COPY INTO

--   agarra los archivos JSONL del volumen y los convierte en filas de una tabla Delta. Es INCREMENTAL: guarda qué archivos ya cargó, por eso re-ejecutarlo NO duplica datos.

-- ¿Por qué 'mergeSchema' aparece DOS veces?
--   FORMAT_OPTIONS: fusiona el schema entre los ARCHIVOS DE ORIGEN. Si hoy llega un archivo con una columna nueva y otro sin ella, se combinan para leer todo.
--   COPY_OPTIONS: evoluciona el schema de la TABLA DESTINO (la Delta).
--     Si el archivo trae una columna que la tabla no tiene, la tabla la AGREGA automáticamente en vez de fallar.

-- Paso 1: crear las tablas SIN columnas. COPY INTO va a inferir las columnas a partir del JSON en la primera carga.

-- ---------- advertisers ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_advertisers;

COPY INTO dev.bronze_ads.bronze_advertisers
FROM '/Volumes/dev/bronze_ads/landing_ads/advertisers'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')   -- leer: fusiona schemas entre archivos origen
COPY_OPTIONS ('mergeSchema' = 'true');   -- escribir: evoluciona el schema de la tabla destino

-- ---------- campaigns ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_campaigns;

COPY INTO dev.bronze_ads.bronze_campaigns
FROM '/Volumes/dev/bronze_ads/landing_ads/campaigns'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- ---------- ads ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_ads;

COPY INTO dev.bronze_ads.bronze_ads
FROM '/Volumes/dev/bronze_ads/landing_ads/ads'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- ---------- impressions ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_impressions;

COPY INTO dev.bronze_ads.bronze_impressions
FROM '/Volumes/dev/bronze_ads/landing_ads/impressions'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- ---------- clicks ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_clicks;

COPY INTO dev.bronze_ads.bronze_clicks
FROM '/Volumes/dev/bronze_ads/landing_ads/clicks'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

-- ---------- conversions ----------
CREATE TABLE IF NOT EXISTS dev.bronze_ads.bronze_conversions;

COPY INTO dev.bronze_ads.bronze_conversions
FROM '/Volumes/dev/bronze_ads/landing_ads/conversions'
FILEFORMAT = JSON
FORMAT_OPTIONS ('mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
8,8,0


In [0]:
-- Verificación: contar cuántas filas quedó en cada tabla bronze.
-- Deberían coincidir con los registros que aterrizamos en el volumen.
SELECT 'advertisers' AS tabla, COUNT(*) AS filas FROM dev.bronze_ads.bronze_advertisers
UNION ALL SELECT 'campaigns',    COUNT(*) FROM dev.bronze_ads.bronze_campaigns
UNION ALL SELECT 'ads',          COUNT(*) FROM dev.bronze_ads.bronze_ads
UNION ALL SELECT 'impressions',  COUNT(*) FROM dev.bronze_ads.bronze_impressions
UNION ALL SELECT 'clicks',       COUNT(*) FROM dev.bronze_ads.bronze_clicks
UNION ALL SELECT 'conversions',  COUNT(*) FROM dev.bronze_ads.bronze_conversions;

tabla,filas
advertisers,400
campaigns,4000
ads,1000000
impressions,10052
clicks,200
conversions,16
